# Class 13: Where You Can Park
*Elements of Data Science — Honors*

**First thing:** save this notebook under a new name that includes your team's name.

Some code is written for you as a worked model. Every `...` is yours. Keep the worksheet
next to you.

In [ ]:
from datascience import *
import numpy as np
# import for plotting
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
# Fix for datascience plots
import collections as collections
import collections.abc as abc
collections.Iterable = abc.Iterable

---
## Part 1. Two Words That Change the Answer

The marble contest from Lab 05: a bag holds two red, two green, and two blue marbles. Draw
three. You win if all three are different colors.

In [ ]:
marbles = make_array('red', 'red', 'blue', 'blue', 'green', 'green')

np.random.choice(marbles, 3)

Run that a few times. Notice what the call does *not* say: whether each marble is put
back in the bag before the next draw. `np.random.choice` has an opinion about that, and it
did not ask you.

So we will ask ourselves. The function below plays one contest and reports whether we won.
Its argument, `put_it_back`, is not a number or a string — it is `True` or `False`, and it
gets handed straight through to `np.random.choice`.

In [ ]:
def three_different_colors(put_it_back):
    """Play one contest. put_it_back is True or False.

    True  = each marble goes back in the bag before the next draw
    False = each marble stays out, the way you would really do it
    """
    draw = np.random.choice(marbles, 3, replace=put_it_back)
    return len(np.unique(draw)) == 3

Try it both ways before you simulate anything. Each call plays one contest and answers
`True` if you won it, so run this cell several times — the answers should change.

In [ ]:
print('put it back:  ', three_different_colors(True))
print('keep it out:  ', three_different_colors(False))

**1.1** Worked model — ten thousand contests, marble **put back** every time.

In [ ]:
wins = 0

for i in np.arange(10000):
    if three_different_colors(True):
        wins = wins + 1

wins / 10000

**1.2** Now ten thousand contests the way you would really reach into a bag. Exactly
one character changes.

In [ ]:
wins = 0

for i in np.arange(10000):
    if ...:
        wins = wins + 1

wins / 10000

**1.3** Record both numbers on the worksheet. One keyword changed the answer by a lot.
Which of the two matches the contest as described? And which one has `np.random.choice`
been doing all along, without being asked?

---
## Part 2. A Stream With No Name

Everything below is **invented**. There is no such creek, nobody measured anything, and the
numbers came out of a random number generator. That is deliberate: to study a *sampling
method* you need a case where you already know the answer, which is exactly why the tank
problem invented an enemy with 1000 tanks.

The table holds all 240 hundred-metre reaches of a 24 km stretch of a simulated stream, with
the chloride concentration of every one. Chloride in a real urban stream does come largely
from road salt, and that much is not invented.

In the field you never have this table. Today you do.

- `km downstream` — distance from the headwaters
- `Cl (mg/L)` — chloride, mostly from road salt washing off pavement
- `access` — `easy` if you can park and walk to the water, `hard` if you cannot

In [ ]:
creek = Table.read_table('data/stream_sim.csv')
creek.show(5)

In [ ]:
creek.num_rows

**2.1** Because we have the whole population, we can compute the thing everyone else
has to estimate. Record it on the worksheet.

In [ ]:
truth = np.mean(creek.column('Cl (mg/L)'))
truth

### Sampling twelve reaches at random

Wading into a creek is slow work. Twelve sites is a realistic day.

`Table.sample` has a default too, and it is `with_replacement=True`. So we say what we
mean.

In [ ]:
one_sample = creek.sample(12, with_replacement=False)
np.mean(one_sample.column('Cl (mg/L)'))

**2.2** Run that a few times and watch the estimate move.

Now look at what the default would have done. Sample without saying
`with_replacement=False`, and count how many *distinct* reaches came back.

In [ ]:
careless = creek.sample(12)

len(np.unique(careless.column('reach')))

Usually that says 12 and nothing looks wrong, which is the problem — the flaw is only
visible across many samples. So count them the way you count anything else: in a loop.

In [ ]:
repeats = 0

for i in np.arange(1000):
    careless = creek.sample(12)
    if len(np.unique(careless.column('reach'))) < 12:
        repeats = repeats + 1

repeats / 1000

**2.3** Record that fraction on the worksheet. Then say what it would have meant, out
in the field, to walk to the same spot twice and write the number down twice.

### A thousand field seasons

Worked model: a thousand honest random samples of twelve.

In [ ]:
random_estimates = make_array()

for i in np.arange(1000):
    s = creek.sample(12, with_replacement=False)
    random_estimates = np.append(random_estimates, np.mean(s.column('Cl (mg/L)')))

print('estimates collected:', len(random_estimates), '  (should be 1000)')
print('their average:      ', np.round(np.mean(random_estimates), 1))

### The sample you would actually collect

Nobody wades 24 km. You drive to where you can park, and you sample the creek there. That
is not laziness — it is what fieldwork costs.

**2.4** Build a thousand convenience samples of **forty** reaches, drawn only from the ones
with `access` equal to `easy`. Forty, not twelve — three times the effort of the honest
survey.

In [ ]:
easy = creek.where('access', 'easy')
easy.num_rows

In [ ]:
convenience_estimates = make_array()

for i in np.arange(1000):
    s = ...
    convenience_estimates = np.append(convenience_estimates, ...)

print('estimates collected:', len(convenience_estimates), '  (should be 1000)')
print('their average:      ', np.round(np.mean(convenience_estimates), 1))

### Both at once

Same bins, same axes, with the truth marked.

**Check first:** both arrays need exactly 1000 estimates. Not 0, and not 1001 — a count
that is *close* to 1000 usually means the loop appended to the wrong array, and the average
it printed belongs to the other simulation. If the cell below reports a *column length
mismatch*, that is the same problem announcing itself late.

In [ ]:
bins = np.arange(60, 141, 2.5)

estimates = Table().with_columns(
    'random n=12', random_estimates,
    'convenience n=40', convenience_estimates
)

estimates.hist(bins=bins)
plt.axvline(truth, color='black', linewidth=2)
plt.title('Estimates of mean chloride — the black line is the truth');

**2.5** Record the centre and the spread of each on the worksheet.

In [ ]:
for name in estimates.labels:
    e = estimates.column(name)
    print(name)
    print('   mean of estimates ', np.round(np.mean(e), 1))
    print('   SD                ', np.round(np.std(e), 1))
    print('   within 10 of truth', np.round(np.mean(np.abs(e - truth) <= 10), 3))

---
## Part 3. What the Shapes Say

**3.1** The two histograms above are of *estimates*. Now look at the chloride values
themselves — the whole creek against the part you can reach.

In [ ]:
creek.hist('Cl (mg/L)', group='access', bins=np.arange(0, 251, 15))

**3.2** Same data as a box plot. Tables want columns of equal length and our two
groups are different sizes, so we hand the job to seaborn, as in the Class 12 warm-up.

In [ ]:
import pandas as pd
import seaborn as sns

df = creek.to_df()

plt.figure(figsize=(4, 5))
sns.boxplot(x='access', y='Cl (mg/L)', data=df)
plt.title('Chloride by accessibility');

**3.3** The `easy` histogram has two humps. The `easy` box plot has one box. Find the
two humps in the data and say what each one is — the `km downstream` column and the
locations of the road crossings will tell you.

Answer on the worksheet, along with what the box plot threw away.

In [ ]:
...

---
That is the whole activity. The discussion questions on the worksheet are the point of it.